# Smart MCQ — Quick Demo Submission (No Training)

**Goal:** Get a valid `submission.csv` on the leaderboard fast, with zero training time.

**Approach:** Use a model that's already pretrained for natural language inference (NLI / entailment) — `cross-encoder/nli-deberta-v3-small`. It's already fine-tuned on millions of (premise, hypothesis) pairs to score "does this premise support this hypothesis."

We treat each option as a hypothesis and the question as a premise. No fine-tuning loop, no GPU training — just forward passes. This typically runs in **2–5 minutes for 500 questions** on a T4, versus 30+ minutes for fine-tuning.

This is a **baseline to secure your leaderboard spot**. It won't hit 75+ MAP@3 (expect roughly 0.45–0.60), but it gets you registered with a real score today. The fine-tuned DeBERTa notebook (separate file) is what you run afterward to push the score up.

In [ ]:
# ── Install (lightweight) ────────────────────────────────────────────────────
# Do NOT reinstall torch — use the Kaggle image's preinstalled, GPU-matched build.
!pip install -q sentencepiece==0.2.0
print('Ready')

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
print(f'torch  : {torch.__version__}')
print(f'cuda   : {torch.cuda.is_available()}')

In [ ]:
# ── GPU smoke test (catches the 'no kernel image' error before it wastes time) ──
GPU_OK = False
if torch.cuda.is_available():
    try:
        _t = torch.randn(8, 8, device='cuda')
        _ = (_t @ _t).sum().item()
        GPU_OK = True
        print(f'✓ GPU usable: {torch.cuda.get_device_name(0)}')
    except RuntimeError as e:
        print(f'✗ GPU kernel test failed ({e}); using CPU instead')
else:
    print('No GPU detected; using CPU')

DEVICE = torch.device('cuda' if GPU_OK else 'cpu')
print(f'Using device: {DEVICE}')
# Note: even on CPU, this notebook is fast because there's no training loop —
# only one forward pass per (question, option) pair.

In [ ]:
CFG = dict(
    test_path   = '/kaggle/input/datasets/kirandwippaul/mcqdata/test.csv',   # ← update slug
    output_path = '/kaggle/working/submission.csv',
    model_name  = 'cross-encoder/nli-deberta-v3-small',  # pretrained NLI, no fine-tune needed
    max_len     = 256,
    batch_size  = 16,         # questions per batch (each → 5 forward passes)
    options     = ['A', 'B', 'C', 'D', 'E'],
)
print('Config ready')
for k, v in CFG.items():
    print(f'  {k:12s}: {v}')

In [ ]:
# ── Load test data ────────────────────────────────────────────────────────────
test_df = pd.read_csv(CFG['test_path'])

# Strip whitespace from column names (common CSV export issue)
test_df.columns = test_df.columns.str.strip()

# Drop fully-empty rows that sometimes appear at the end of Kaggle CSVs
test_df = test_df.dropna(how='all').reset_index(drop=True)

for col in CFG['options']:
    test_df[col] = test_df[col].fillna('').astype(str)

print(f'Test rows   : {len(test_df)}')
print(f'Columns     : {list(test_df.columns)}')
print(f'ID sample   : {test_df.iloc[:3, 0].tolist()}')
print()
test_df.head(3)


In [ ]:
# ── Load pretrained NLI model — already trained, no fine-tuning needed ─────────
print(f'Loading {CFG["model_name"]} ...')
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
model     = AutoModelForSequenceClassification.from_pretrained(CFG['model_name'])
model.to(DEVICE)
model.eval()

# NLI models output 3 classes: contradiction, entailment, neutral (order varies by model)
print('Label mapping:', model.config.id2label)

# Find which output index corresponds to 'entailment' — that's our "this option
# is supported by / answers the question" signal.
label2id = {v.lower(): k for k, v in model.config.id2label.items()}
ENTAIL_IDX = label2id.get('entailment', 1)  # fallback to index 1 if unexpected labels
print(f'Using entailment logit index: {ENTAIL_IDX}')
print('✓ Model ready — no training required')

In [ ]:
# ── Scoring function ──────────────────────────────────────────────────────────
@torch.inference_mode()
def score_batch(questions, options_batch):
    """
    questions      : list[str], length B
    options_batch  : list[list[str]], shape (B, 5)

    Returns
    -------
    entail_probs : np.array, shape (B, 5) — entailment probability per option
    """
    B = len(questions)
    # Flatten into B*5 (premise, hypothesis) pairs
    premises    = []
    hypotheses  = []
    for q, opts in zip(questions, options_batch):
        for opt in opts:
            premises.append(q)
            # Phrase the option as a claim the question would support
            hypotheses.append(f'The correct answer is: {opt}')

    enc = tokenizer(
        premises, hypotheses,
        truncation=True, max_length=CFG['max_len'],
        padding=True, return_tensors='pt',
    ).to(DEVICE)

    logits = model(**enc).logits              # (B*5, 3)
    probs  = torch.softmax(logits, dim=-1)     # (B*5, 3)
    entail = probs[:, ENTAIL_IDX]              # (B*5,)

    return entail.cpu().numpy().reshape(B, 5)


# Quick sanity check on first row
sample = test_df.iloc[0]
sample_opts = [sample[o] for o in CFG['options']]
test_scores = score_batch([sample['prompt']], [sample_opts])
print('Sample entailment scores (A,B,C,D,E):', np.round(test_scores[0], 4))
print('Top pick:', CFG['options'][test_scores[0].argmax()])

In [ ]:
# ── Run inference on full test set ────────────────────────────────────────────
# NOTE: all_scores is reset here — safe to re-run this cell without duplicating
all_scores = []   # ← reset every time this cell runs
N  = len(test_df)
bs = CFG['batch_size']

for start in tqdm(range(0, N, bs), desc='Scoring questions'):
    chunk        = test_df.iloc[start : start + bs]
    questions    = chunk['prompt'].tolist()
    options_list = [[row[o] for o in CFG['options']] for _, row in chunk.iterrows()]

    scores = score_batch(questions, options_list)   # (b, 5)
    all_scores.append(scores)

all_scores = np.concatenate(all_scores, axis=0)    # (N, 5)
assert len(all_scores) == len(test_df), \
    f'Score count {len(all_scores)} != test rows {len(test_df)}'
print(f'Done. Scores shape: {all_scores.shape}  ← must be ({N}, 5)')


In [ ]:
# ── Build submission.csv ─────────────────────────────────────────────────────
import re

idx2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

top3_idx = np.argsort(all_scores, axis=-1)[:, ::-1][:, :3]   # (N, 3)
predictions = [' '.join(idx2label[i] for i in row) for row in top3_idx]

# Auto-detect id column name
id_col = next((c for c in test_df.columns if c.lower() == 'id'), None)
ids    = test_df[id_col].tolist() if id_col else list(range(1, len(test_df) + 1))

# Validate all predictions have correct format
for i, p in enumerate(predictions):
    if not re.match(r'^[A-E] [A-E] [A-E]$', p):
        raise ValueError(f'Row {i} malformed: {repr(p)}')

submission = pd.DataFrame({'id': ids, 'Prediction': predictions})

EXPECTED = len(test_df)
assert len(submission) == EXPECTED, \
    f'Expected {EXPECTED} rows, got {len(submission)} — re-run the scoring cell above'

submission.to_csv(CFG['output_path'], index=False)

# Read back to confirm
readback = pd.read_csv(CFG['output_path'])
print(f'Rows in file : {len(readback)}  ← must be {EXPECTED}')
print(f'Columns      : {list(readback.columns)}')
print()
print(submission.head(10).to_string(index=False))
print('\nFirst-choice distribution:')
print(submission['Prediction'].str[0].value_counts().sort_index())


## Submit now

1. `submission.csv` is in the Output panel (right side)
2. Click **Submit** at the top of the notebook, or go to the competition page → **Submit Predictions**
3. This secures your registration with a valid score — typically **MAP@3 ≈ 0.45–0.60** with zero training

## After registration is locked in

Switch to the fine-tuned `deberta-v3-small` option-wise notebook (the one with the training loop) to push toward 75+. That one needs ~25–35 min of actual training but gets you real gains:

| Notebook | Time | MAP@3 |
|---|---|---|
| **This one (pretrained NLI, no training)** | ~3–5 min | ~0.45–0.60 |
| Fine-tuned deberta-v3-small | ~30 min | ~0.65–0.75 |
| Fine-tuned deberta-v3-base | ~45 min | ~0.72–0.80 |

If training felt slow, two quick speedups for the fine-tuning notebook:
- Lower `max_len` from 512 → 256 (often 2x faster, minor accuracy cost)
- Lower `epochs` from 3 → 2 (DeBERTa-small converges fast; epoch 1→2 has the biggest jump)